In [1]:
"""
Home Credit Default Risk — table merging & aggregation pipeline.

Converts the relational tables (bureau, credit_card_balance,
installments_payments, POS_CASH_balance, previous_application) into
customer-level features and merges them all into a single app_train frame.

Run order matters: each section deletes its raw dataframe after merging
to keep peak memory down — don't reorder sections without checking
which variables get del'd along the way.
"""

import gc

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

DATA_DIR = r"C:\Users\Mahmoud\Desktop\Project - Copy\data"

# STEP 0 — Load application_train / application_test, stack them
#          with an is_train flag so feature engineering on app_train
#          downstream applies identically to the test set
def load_applications():
    app_test = pd.read_csv(rf"{DATA_DIR}\application_test.csv")
    app_train = pd.read_csv(rf"{DATA_DIR}\application_train.csv")

    app_train['is_train'] = 1
    app_test['is_train'] = 0

    app_train = pd.concat([app_train, app_test], axis=0, ignore_index=True)

    del app_test
    gc.collect()
    return app_train

# STEP 1 — Bureau + bureau_balance aggregation
def aggregate_bureau(app_train):
    bureau = pd.read_csv(rf"{DATA_DIR}\bureau.csv")
    bureau_bal = pd.read_csv(rf"{DATA_DIR}\bureau_balance.csv")

    # 1a. Bureau balance aggregation & merge into bureau
    bureau_bal_encoded = pd.get_dummies(bureau_bal, columns=['STATUS'])

    bb_agg = bureau_bal_encoded.groupby('SK_ID_BUREAU').agg({
        'MONTHS_BALANCE': ['min', 'max', 'size'],
        'STATUS_0': ['mean'],
        'STATUS_1': ['mean'],
        'STATUS_2': ['mean'],
        'STATUS_3': ['mean'],
        'STATUS_4': ['mean'],
        'STATUS_5': ['mean'],
        'STATUS_C': ['mean'],
        'STATUS_X': ['mean'],
    })
    bb_agg.columns = pd.Index(
        ['BB_' + e[0] + '_' + e[1].upper() for e in bb_agg.columns.tolist()]
    )
    bb_features = list(bb_agg.columns)

    bureau = bureau.merge(bb_agg, how='left', on='SK_ID_BUREAU')

    del bureau_bal, bureau_bal_encoded, bb_agg
    gc.collect()

    # 1b. One-hot encode bureau categoricals
    categorical_cols = bureau.select_dtypes(include=['object']).columns
    original_cols = set(bureau.columns)
    bureau_encoded = pd.get_dummies(bureau, columns=categorical_cols)
    dummy_cols = [c for c in bureau_encoded.columns if c not in original_cols]

    # 1c. Aggregation spec — numeric + bureau_balance + one-hot
    num_aggregations = {
        'DAYS_CREDIT': ['min', 'max', 'mean'],
        'CREDIT_DAY_OVERDUE': ['max', 'mean'],
        'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
        'DAYS_ENDDATE_FACT': ['min', 'max', 'mean'],
        'AMT_CREDIT_MAX_OVERDUE': ['max', 'mean'],
        'CNT_CREDIT_PROLONG': ['sum', 'mean'],
        'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_LIMIT': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_OVERDUE': ['mean'],
        'DAYS_CREDIT_UPDATE': ['min', 'max', 'mean'],
        'AMT_ANNUITY': ['max', 'mean', 'sum'],
    }

    for col in bb_features:
        if col.endswith('_MIN'):
            num_aggregations[col] = ['min']
        elif col.endswith('_MAX'):
            num_aggregations[col] = ['max']
        elif col.endswith('_SIZE'):
            num_aggregations[col] = ['sum']
        else:
            num_aggregations[col] = ['mean']

    cat_aggregations = {col: ['mean'] for col in dummy_cols}
    bureau_aggregations = {**num_aggregations, **cat_aggregations}

    # 1d. Group, rename, merge
    bureau_agg = bureau_encoded.groupby('SK_ID_CURR').agg(bureau_aggregations)
    bureau_agg.columns = pd.Index(
        ['BUREAU_' + e[0] + '_' + e[1].upper() for e in bureau_agg.columns.tolist()]
    )
    bureau_agg['has_bureau'] = 1

    app_train = app_train.merge(bureau_agg, how='left', on='SK_ID_CURR')
    app_train['has_bureau'] = app_train['has_bureau'].fillna(0).astype(int)

    del bureau, bureau_encoded, bureau_agg
    gc.collect()
    return app_train

# STEP 2 — Credit card balance aggregation
def aggregate_credit_card(app_train):
    cc_bal = pd.read_csv(rf"{DATA_DIR}\credit_card_balance.csv")

    categorical_cols = cc_bal.select_dtypes(include=['object']).columns
    cc_bal_encoded = pd.get_dummies(cc_bal, columns=categorical_cols)

    num_aggregations = {
        'AMT_BALANCE': ['mean', 'max'],
        'AMT_CREDIT_LIMIT_ACTUAL': ['mean'],
        'AMT_PAYMENT_CURRENT': ['mean'],
        'MONTHS_BALANCE': ['min', 'max', 'size'],
    }
    cat_aggregations = {
        col: ['mean']
        for col in cc_bal_encoded.columns
        if col.startswith('NAME_CONTRACT_STATUS_')
    }
    cc_bal_aggregations = {**num_aggregations, **cat_aggregations}

    cc_bal_agg = cc_bal_encoded.groupby('SK_ID_CURR').agg(cc_bal_aggregations)
    cc_bal_agg.columns = pd.Index(
        ['Credit_card_' + e[0] + '_' + e[1].upper() for e in cc_bal_agg.columns.tolist()]
    )
    cc_bal_agg['has_credit_card'] = 1

    app_train = app_train.merge(cc_bal_agg, how='left', on='SK_ID_CURR')
    app_train['has_credit_card'] = app_train['has_credit_card'].fillna(0).astype(int)

    del cc_bal, cc_bal_encoded
    gc.collect()
    return app_train

# STEP 3 — Installments payments (full history + recent 365 days)
def aggregate_installments(app_train):
    ins = pd.read_csv(rf"{DATA_DIR}\installments_payments.csv")

    # 3a. Engineered per-row features
    payment_delay = ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']
    ins['ins_DPD'] = payment_delay.clip(lower=0)
    ins['ins_DBD'] = (-payment_delay).clip(lower=0)
    ins['ins_PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    ins['ins_PAYMENT_RATIO'] = ins['AMT_PAYMENT'] / (ins['AMT_INSTALMENT'] + 1e-5)

    # 3b. Full-history aggregation
    full_aggregations = {
        'NUM_INSTALMENT_VERSION': ['nunique'],
        'NUM_INSTALMENT_NUMBER': ['max'],
        'DAYS_INSTALMENT': ['min', 'max'],
        'AMT_INSTALMENT': ['max', 'mean', 'sum'],
        'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
        'ins_DPD': ['max', 'mean', 'sum'],
        'ins_DBD': ['max', 'mean', 'sum'],
        'ins_PAYMENT_DIFF': ['max', 'mean', 'sum'],
        'ins_PAYMENT_RATIO': ['mean', 'max'],
        'DAYS_ENTRY_PAYMENT': ['size'],
    }

    ins_full_agg = ins.groupby('SK_ID_CURR').agg(full_aggregations)
    ins_full_agg.columns = pd.Index(
        ['Installments_' + e[0] + '_' + e[1].upper() for e in ins_full_agg.columns.tolist()]
    )
    ins_full_agg['has_installments'] = 1

    app_train = app_train.merge(ins_full_agg, how='left', on='SK_ID_CURR')
    app_train['has_installments'] = app_train['has_installments'].fillna(0).astype(int)

    del ins_full_agg
    gc.collect()

    # 3c. Recent-history aggregation (last 365 days only)
    ins_recent = ins[ins['DAYS_INSTALMENT'] >= -365].copy()

    recent_aggregations = {
        'ins_DPD': ['max', 'mean', 'sum'],
        'ins_DBD': ['max', 'mean'],
        'ins_PAYMENT_DIFF': ['max', 'mean', 'sum'],
        'ins_PAYMENT_RATIO': ['mean', 'max'],
        'AMT_PAYMENT': ['sum', 'mean'],
        'DAYS_ENTRY_PAYMENT': ['size'],
    }

    ins_recent_agg = ins_recent.groupby('SK_ID_CURR').agg(recent_aggregations)
    ins_recent_agg.columns = pd.Index(
        ['Installments_RECENT_' + e[0] + '_' + e[1].upper()
         for e in ins_recent_agg.columns.tolist()]
    )

    app_train = app_train.merge(ins_recent_agg, how='left', on='SK_ID_CURR')

    del ins, ins_recent, ins_recent_agg
    gc.collect()
    return app_train


# STEP 4 — POS_CASH_balance aggregation
def aggregate_pos_cash(app_train):
    pos = pd.read_csv(rf"{DATA_DIR}\POS_CASH_balance.csv")

    pos['INSTALLMENTS_PAID'] = pos['CNT_INSTALMENT'] - pos['CNT_INSTALMENT_FUTURE']
    pos['LOAN_PERCENT_REMAINING'] = pos['CNT_INSTALMENT_FUTURE'] / (pos['CNT_INSTALMENT'] + 1e-5)

    pos_encoded = pd.get_dummies(pos, columns=['NAME_CONTRACT_STATUS'])

    agg = {
        'MONTHS_BALANCE': ['min', 'max', 'size'],
        'SK_DPD': ['max', 'mean'],
        'SK_DPD_DEF': ['max', 'mean'],
        'INSTALLMENTS_PAID': ['mean'],
        'LOAN_PERCENT_REMAINING': ['mean'],
    }

    dummy_cols = [c for c in pos_encoded.columns if c.startswith('NAME_CONTRACT_STATUS_')]
    for col in dummy_cols:
        agg[col] = ['mean']

    pos_agg = pos_encoded.groupby('SK_ID_CURR').agg(agg)
    pos_agg.columns = pd.Index(
        ['POS_' + e[0] + '_' + e[1].upper() for e in pos_agg.columns.tolist()]
    )
    pos_agg['has_pos_cash'] = 1

    app_train = app_train.merge(pos_agg, how='left', on='SK_ID_CURR')
    app_train['has_pos_cash'] = app_train['has_pos_cash'].fillna(0).astype(int)

    del pos, pos_encoded, pos_agg
    gc.collect()
    return app_train

# STEP 5 — previous_application aggregation
#          (lowercase filename only — the uppercase read in the
#          original notebook was dead code that always 404s on
#          case-sensitive filesystems, so it's removed here)
def aggregate_previous_application(app_train):
    prev = pd.read_csv(rf"{DATA_DIR}\previous_application.csv")

    # 5a. Sentinel cleanup (365243 = "no date" placeholder)
    days_cols = [
        'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE',
        'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION',
    ]
    prev[days_cols] = prev[days_cols].replace(365243, np.nan)

    # 5b. Row-level ratio features
    prev['CREDIT_TO_APPLICATION_RATIO'] = (
        prev['AMT_CREDIT'] / prev['AMT_APPLICATION'].replace(0, np.nan)
    )
    prev['DOWN_PAYMENT_RATIO'] = (
        prev['AMT_DOWN_PAYMENT'] / prev['AMT_GOODS_PRICE'].replace(0, np.nan)
    )
    prev['CREDIT_TO_GOODS_RATIO'] = (
        prev['AMT_CREDIT'] / prev['AMT_GOODS_PRICE'].replace(0, np.nan)
    )
    prev['ANNUITY_TO_CREDIT_RATIO'] = (
        prev['AMT_ANNUITY'] / prev['AMT_CREDIT'].replace(0, np.nan)
    )

    # 5c. Time-span features
    prev['LOAN_DURATION_PLANNED'] = prev['DAYS_LAST_DUE'] - prev['DAYS_FIRST_DUE']
    prev['DAYS_LAST_DUE_DIFF'] = prev['DAYS_LAST_DUE'] - prev['DAYS_LAST_DUE_1ST_VERSION']
    prev['DAYS_DECISION_TO_DRAWING'] = prev['DAYS_FIRST_DRAWING'] - prev['DAYS_DECISION']

    # 5d. Boolean / flag features
    prev['WAS_APPROVED'] = (prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
    prev['WAS_REFUSED'] = (prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)
    prev['WAS_CANCELED'] = (prev['NAME_CONTRACT_STATUS'] == 'Canceled').astype(int)
    prev['APPLICATION_GT_CREDIT'] = (prev['AMT_APPLICATION'] > prev['AMT_CREDIT']).astype(int)
    prev['LOAN_WAS_EXTENDED'] = (prev['DAYS_LAST_DUE_DIFF'] > 0).astype(int)

    # 5e. One-hot encode categoricals
    cat_cols = [
        'NAME_CONTRACT_TYPE', 'NAME_CONTRACT_STATUS',
        'NAME_CASH_LOAN_PURPOSE', 'NAME_PAYMENT_TYPE',
        'CODE_REJECT_REASON', 'NAME_CLIENT_TYPE',
        'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO',
        'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE',
        'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP',
        'PRODUCT_COMBINATION', 'NAME_TYPE_SUITE',
    ]
    original_cols = set(prev.columns)
    prev_encoded = pd.get_dummies(prev, columns=cat_cols, dummy_na=True)
    dummy_cols = [c for c in prev_encoded.columns if c not in original_cols]

    # 5f. Aggregation spec — numeric + one-hot
    numeric_agg = {
        'AMT_ANNUITY': ['mean', 'max', 'sum'],
        'AMT_APPLICATION': ['mean', 'max', 'sum'],
        'AMT_CREDIT': ['mean', 'max', 'sum'],
        'AMT_DOWN_PAYMENT': ['mean', 'sum'],
        'AMT_GOODS_PRICE': ['mean', 'max'],
        'RATE_DOWN_PAYMENT': ['mean', 'max'],
        'RATE_INTEREST_PRIMARY': ['mean', 'max'],
        'RATE_INTEREST_PRIVILEGED': ['mean', 'min'],
        'DAYS_DECISION': ['mean', 'min', 'max'],
        'CNT_PAYMENT': ['mean', 'sum'],
        'CREDIT_TO_APPLICATION_RATIO': ['mean', 'min', 'max'],
        'DOWN_PAYMENT_RATIO': ['mean'],
        'CREDIT_TO_GOODS_RATIO': ['mean', 'max'],
        'ANNUITY_TO_CREDIT_RATIO': ['mean', 'max'],
        'LOAN_DURATION_PLANNED': ['mean', 'max'],
        'DAYS_LAST_DUE_DIFF': ['mean', 'max'],
        'DAYS_DECISION_TO_DRAWING': ['mean'],
        'WAS_APPROVED': ['sum', 'mean'],
        'WAS_REFUSED': ['sum', 'mean'],
        'WAS_CANCELED': ['sum', 'mean'],
        'APPLICATION_GT_CREDIT': ['sum', 'mean'],
        'LOAN_WAS_EXTENDED': ['sum', 'mean'],
        'NFLAG_INSURED_ON_APPROVAL': ['mean'],
        'SELLERPLACE_AREA': ['mean', 'max'],
        'SK_ID_PREV': ['count'],
    }
    for col in dummy_cols:
        numeric_agg[col] = ['mean']

    prev_agg = prev_encoded.groupby('SK_ID_CURR').agg(numeric_agg)
    prev_agg.columns = ['PREV_' + col[0] + '_' + col[1].upper() for col in prev_agg.columns]
    prev_agg = prev_agg.reset_index()
    prev_agg.rename(columns={'PREV_SK_ID_PREV_COUNT': 'PREV_TOTAL_APPS'}, inplace=True)

    # 5g. Recency features — most recent application per client
    prev_sorted = prev.sort_values('DAYS_DECISION', ascending=False)
    last_app = prev_sorted.groupby('SK_ID_CURR').first().reset_index()

    recency_cols = [
        'AMT_CREDIT', 'AMT_APPLICATION', 'AMT_ANNUITY',
        'NAME_CONTRACT_STATUS', 'NAME_YIELD_GROUP',
        'CNT_PAYMENT', 'WAS_APPROVED', 'WAS_REFUSED',
        'DAYS_DECISION', 'CREDIT_TO_APPLICATION_RATIO',
    ]
    last_app_subset = last_app[['SK_ID_CURR'] + recency_cols].copy()
    last_app_subset.columns = ['SK_ID_CURR'] + [f'PREV_LAST_{c}' for c in recency_cols]

    prev_agg = prev_agg.merge(last_app_subset, on='SK_ID_CURR', how='left')

    # 5h. Merge into master, clean up
    app_train = app_train.merge(prev_agg, on='SK_ID_CURR', how='left')

    del prev, prev_encoded, prev_agg, prev_sorted, last_app, last_app_subset
    gc.collect()
    return app_train


# MAIN — run the full merge pipeline end to end
def main():
    app_train = load_applications()
    app_train = aggregate_bureau(app_train)
    app_train = aggregate_credit_card(app_train)
    app_train = aggregate_installments(app_train)
    app_train = aggregate_pos_cash(app_train)
    app_train = aggregate_previous_application(app_train)

    cols_df = pd.DataFrame({'column_name': app_train.columns})
    print(cols_df.to_string())
    print(f"\nFinal merged shape: {app_train.shape}")

    # Fixed: original notebook saved with no .parquet extension
    output_path = rf"{DATA_DIR}\app_train_merged.parquet"
    app_train.to_parquet(output_path, index=False)
    print(f"Saved to: {output_path}")

    return app_train


if __name__ == '__main__':
    app_train = main()

                                                               column_name
0                                                               SK_ID_CURR
1                                                                   TARGET
2                                                       NAME_CONTRACT_TYPE
3                                                              CODE_GENDER
4                                                             FLAG_OWN_CAR
5                                                          FLAG_OWN_REALTY
6                                                             CNT_CHILDREN
7                                                         AMT_INCOME_TOTAL
8                                                               AMT_CREDIT
9                                                              AMT_ANNUITY
10                                                         AMT_GOODS_PRICE
11                                                         NAME_TYPE_SUITE
12                       